# Подготовка датасета для задачи определения новизны новостей

Цель ноутбука — провести первичный анализ новостных данных и подготовить датасет для дальнейшего решения задачи **semantic news novelty detection**.

В рамках проекта новость рассматривается как элемент временного новостного потока. Система должна группировать публикации по сюжетам и определять, добавляет ли новая публикация существенную информацию к уже известному контексту сюжета.

В ноутбуке выполняются:

1. описание источника и состава данных;
2. базовый EDA, важный для моделирования;
3. проверка качества данных и базовая очистка;
4. анализ пропусков, дублей, длины текстов, источников и временного распределения;
5. описание подхода к разметке и оценке её качества;
6. описание алгоритма формирования выборки;
7. описание стратегии валидации;
8. сохранение подготовленного датасета.

> Важно: ноутбук не углубляется в feature engineering. Основной фокус — анализ исходных данных и подготовка корректной выборки для последующих экспериментов.

## 1. Постановка задачи

В проекте используется следующая постановка:

> Для новой публикации внутри существующего сюжетного кластера определить, является ли она существенным обновлением или в основном повторяет уже известную информацию.

Для первой версии задачи целевую переменную удобно задать бинарно:

- `label = 1` — публикация содержит существенное обновление / новую информацию;
- `label = 0` — публикация является дублем, рерайтом или несущественным обновлением.

Основная ML-часть проекта — построение семантического пространства новостей с помощью embedding-модели. Дальше поверх эмбеддингов рассчитываются similarity-признаки, которые позволяют интерпретируемо принимать решения о принадлежности новости к сюжету и степени её новизны.

Общий пайплайн проекта:

```text
news text
→ semantic embedding model
→ similarity search
→ incremental story clustering
→ novelty scoring
→ decision layer
```

## 2. Источник и состав данных

В качестве исходных данных используется датасет новостных публикаций на русском языке. Каждая запись соответствует одной публикации и содержит текст новости, заголовок, дату публикации, источник и служебные поля.

Ожидаемые поля исходного датасета:

- `news_id` — уникальный идентификатор новости;
- `title` — заголовок новости;
- `text` — полный текст публикации;
- `published_at` — дата и время публикации;
- `source` — источник публикации;
- `url` — ссылка на публикацию, если доступна.

Если фактические названия колонок отличаются, их можно указать в конфигурационном блоке ниже.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

# ---------------------------------------------------------------------
# Конфигурация ноутбука
# ---------------------------------------------------------------------

DATA_PATH = Path("data/news.csv")
OUTPUT_PATH = Path("data/processed/news_novelty_dataset.csv")

# Если в вашем датасете колонки называются иначе, измените значения здесь.
ID_COL = "news_id"
TITLE_COL = "title"
TEXT_COL = "text"
DATE_COL = "published_at"
SOURCE_COL = "source"
URL_COL = "url"

# Минимальная длина текста для дальнейшей работы.
MIN_TEXT_WORDS = 20

# Параметры будущей логики формирования выборки.
TEMPORAL_WINDOW_DAYS = 14
RECENT_K = 50

## 3. Загрузка данных и первичный обзор

На первом этапе проверяется структура датасета: количество записей, набор столбцов, типы данных и наличие пропусков.

Для дальнейшей модели критически важны поля с текстом и временем публикации:

- текст используется для построения эмбеддингов;
- дата публикации нужна для обработки новостей как временного потока.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Файл {DATA_PATH} не найден. "
        "Укажите корректный путь к исходному датасету в переменной DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)
print(f"Размер исходного датасета: {df.shape[0]:,} строк, {df.shape[1]:,} колонок")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

### Выводы по первичному обзору

На этом этапе нужно зафиксировать:

- сколько строк и колонок содержит датасет;
- присутствуют ли необходимые поля для моделирования;
- какие поля требуют приведения типов;
- есть ли признаки потенциальных проблем: пропуски, дубли, некорректные даты, пустые тексты.

## 4. Проверка наличия обязательных колонок

Перед EDA проверим, что в датасете присутствуют ключевые поля. Если часть полей отсутствует, ноутбук не должен молча продолжать работу с неправильными данными.

In [ ]:
required_cols = [TEXT_COL, DATE_COL]
optional_cols = [ID_COL, TITLE_COL, SOURCE_COL, URL_COL]

missing_required = [col for col in required_cols if col not in df.columns]
missing_optional = [col for col in optional_cols if col not in df.columns]

if missing_required:
    raise ValueError(f"В датасете отсутствуют обязательные колонки: {missing_required}")

print("Обязательные колонки найдены.")
if missing_optional:
    print(f"Необязательные колонки отсутствуют: {missing_optional}")

## 5. Проверка качества и чистоты данных

Для задачи семантической векторизации особенно важны следующие проблемы:

- отсутствующий или пустой текст;
- слишком короткие публикации;
- полные дубли текста;
- некорректные даты публикации;
- сильный перекос по источникам;
- неравномерность новостного потока по времени.

In [ ]:
missing_share = df.isna().mean().sort_values(ascending=False).to_frame("missing_share")
missing_share[missing_share["missing_share"] > 0]

In [ ]:
quality_report = {}

quality_report["rows_total"] = len(df)
quality_report["full_duplicate_rows"] = int(df.duplicated().sum())
quality_report["missing_text"] = int(df[TEXT_COL].isna().sum())
quality_report["empty_text"] = int(df[TEXT_COL].fillna("").str.strip().eq("").sum())

if TITLE_COL in df.columns:
    quality_report["missing_title"] = int(df[TITLE_COL].isna().sum())
    quality_report["empty_title"] = int(df[TITLE_COL].fillna("").str.strip().eq("").sum())

if URL_COL in df.columns:
    quality_report["missing_url"] = int(df[URL_COL].isna().sum())
    quality_report["duplicate_url"] = int(df[URL_COL].dropna().duplicated().sum())

quality_report["duplicate_text"] = int(df[TEXT_COL].dropna().duplicated().sum())

pd.Series(quality_report).to_frame("value")

In [ ]:
# Приведение даты публикации к datetime.
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

print(f"Некорректных дат публикации: {df[DATE_COL].isna().sum():,}")
print(f"Минимальная дата: {df[DATE_COL].min()}")
print(f"Максимальная дата: {df[DATE_COL].max()}")

In [ ]:
# Базовые технические признаки качества текста.
df["text_length"] = df[TEXT_COL].fillna("").astype(str).str.len()
df["text_num_words"] = df[TEXT_COL].fillna("").astype(str).str.split().str.len()

if TITLE_COL in df.columns:
    df["title_length"] = df[TITLE_COL].fillna("").astype(str).str.len()
    df["title_num_words"] = df[TITLE_COL].fillna("").astype(str).str.split().str.len()

df[["text_length", "text_num_words"]].describe()

In [ ]:
# Примеры слишком коротких публикаций.
short_texts = df[df["text_num_words"] < MIN_TEXT_WORDS]
print(f"Публикаций короче {MIN_TEXT_WORDS} слов: {len(short_texts):,}")
cols_to_show = [col for col in [ID_COL, TITLE_COL, TEXT_COL, DATE_COL, SOURCE_COL] if col in df.columns]
short_texts[cols_to_show].head(10)

### Выводы по качеству данных

Для дальнейшего моделирования необходимо удалить или отдельно обработать:

- записи без текста;
- записи с некорректной датой публикации;
- полные дубли текста;
- слишком короткие публикации, так как они могут содержать недостаточно контекста для корректной семантической векторизации.

Дубли по тексту особенно важны для задачи novelty detection: если оставить полные дубли в данных, они могут исказить оценку качества кластеризации и поиска похожих новостей.

## 6. Базовая очистка данных

На этом этапе выполняется только минимальная очистка, необходимая для формирования корректной выборки. Глубокая обработка признаков и сложный feature engineering намеренно не выполняются.

In [ ]:
df_clean = df.copy()
rows_before = len(df_clean)

# Удаляем записи без текста и даты.
df_clean = df_clean.dropna(subset=[TEXT_COL, DATE_COL])
df_clean = df_clean[df_clean[TEXT_COL].astype(str).str.strip() != ""]

# Удаляем полные дубли текста.
df_clean = df_clean.drop_duplicates(subset=[TEXT_COL])

# При наличии URL удаляем дубли ссылок.
if URL_COL in df_clean.columns:
    df_clean = df_clean.drop_duplicates(subset=[URL_COL])

# Удаляем слишком короткие публикации.
df_clean = df_clean[df_clean["text_num_words"] >= MIN_TEXT_WORDS]

# Сортируем по времени публикации.
df_clean = df_clean.sort_values(DATE_COL).reset_index(drop=True)

rows_after = len(df_clean)
print(f"Строк до очистки: {rows_before:,}")
print(f"Строк после очистки: {rows_after:,}")
print(f"Удалено строк: {rows_before - rows_after:,} ({(rows_before - rows_after) / rows_before:.2%})")

In [ ]:
df_clean.head()

### Выводы по очистке

После базовой очистки остаются публикации, пригодные для построения эмбеддингов и дальнейшей обработки в хронологическом порядке.

Сортировка по времени публикации является принципиальной: в будущей системе новости должны обрабатываться как поток, а не как случайно перемешанный набор документов.

## 7. EDA, важный для моделирования

В этом разделе анализируются только те свойства данных, которые влияют на будущую модель и алгоритм формирования выборки:

- длина текстов;
- распределение по источникам;
- распределение по времени;
- наличие дублей;
- базовые корреляции технических числовых признаков.

### 7.1 Распределение длины текстов

Длина текста важна для semantic encoder: слишком короткие тексты содержат мало контекста, а слишком длинные могут обрезаться моделью.

In [ ]:
plt.figure(figsize=(10, 5))
df_clean["text_num_words"].hist(bins=50)
plt.title("Распределение длины текстов в словах")
plt.xlabel("Количество слов")
plt.ylabel("Количество публикаций")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
df_clean["text_length"].hist(bins=50)
plt.title("Распределение длины текстов в символах")
plt.xlabel("Количество символов")
plt.ylabel("Количество публикаций")
plt.show()

In [ ]:
df_clean[["text_length", "text_num_words"]].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

**Вывод для моделирования:** распределение длины текстов позволяет выбрать минимальную длину публикации для включения в выборку и понять, насколько часто тексты могут превышать ограничение embedding-модели на максимальную длину входа.

### 7.2 Распределение по источникам

Распределение по источникам важно для оценки репрезентативности. Если один источник доминирует, модель и правила могут частично подстроиться под стиль конкретного издания.

In [ ]:
if SOURCE_COL in df_clean.columns:
    source_counts = df_clean[SOURCE_COL].value_counts()
    display(source_counts.head(20).to_frame("count"))

    plt.figure(figsize=(12, 6))
    source_counts.head(20).sort_values().plot(kind="barh")
    plt.title("Топ-20 источников по количеству публикаций")
    plt.xlabel("Количество публикаций")
    plt.ylabel("Источник")
    plt.show()
else:
    print("Колонка с источником отсутствует, анализ распределения по источникам пропущен.")

**Вывод для моделирования:** при сильном перекосе по источникам желательно учитывать это при валидации и ручной разметке, чтобы итоговая оценка качества не отражала только поведение системы на одном доминирующем источнике.

### 7.3 Распределение публикаций по времени

Временное распределение важно для выбора temporal window при кластеризации и для стратегии валидации.

In [ ]:
publications_by_day = df_clean.set_index(DATE_COL).resample("D").size()

plt.figure(figsize=(14, 5))
publications_by_day.plot()
plt.title("Количество публикаций по дням")
plt.xlabel("Дата")
plt.ylabel("Количество публикаций")
plt.show()

publications_by_day.describe()

In [ ]:
publications_by_month = df_clean.set_index(DATE_COL).resample("M").size()
publications_by_month.to_frame("count").tail(24)

**Вывод для моделирования:** если поток публикаций неравномерен, temporal split для валидации должен учитывать периоды с достаточным числом наблюдений. Временное окно для активных кластеров также должно подбираться с учётом плотности публикаций.

### 7.4 Корреляции числовых технических признаков

Так как задача является текстовой, классические корреляции по сырым числовым признакам имеют ограниченную ценность. Тем не менее их полезно проверить для контроля качества данных.

In [ ]:
numeric_cols = ["text_length", "text_num_words"]

if "title_length" in df_clean.columns:
    numeric_cols += ["title_length", "title_num_words"]

corr = df_clean[numeric_cols].corr()
corr

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(corr, aspect="auto")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.index)), corr.index)
plt.colorbar(label="Correlation")
plt.title("Корреляция технических числовых признаков")
plt.tight_layout()
plt.show()

**Вывод по корреляциям:** ожидаемо наиболее сильная связь наблюдается между длиной текста в символах и количеством слов. Эти признаки не являются основными предикторами новизны, но полезны для контроля качества данных и поиска аномально коротких или длинных публикаций.

## 8. Нужно ли масштабирование признаков

Классическое масштабирование признаков на этапе работы с сырыми текстами не требуется, так как модель получает на вход текст, а не табличные числовые признаки.

После получения embedding-векторов для корректного расчёта cosine similarity может использоваться **L2-нормализация эмбеддингов**:

```text
vector_normalized = vector / ||vector||
```

Если в дальнейшем поверх similarity-признаков будет обучаться отдельный классификатор, необходимость масштабирования будет зависеть от выбранного алгоритма:

- для rule-based decision layer масштабирование не требуется;
- для логистической регрессии или SVM масштабирование числовых признаков может быть полезно;
- для деревьев решений, RandomForest, CatBoost, XGBoost масштабирование обычно не является обязательным.

В текущей версии проекта решение принимается интерпретируемым rule-based слоем поверх similarity-признаков, поэтому отдельное масштабирование табличных признаков не требуется.

## 9. Алгоритм формирования выборки для задачи

Итоговая выборка должна соответствовать будущему production-сценарию: новости приходят во времени, сравниваются с активными сюжетными кластерами, после чего система оценивает степень новизны.

Единица выборки:

```text
(candidate_news, cluster_context) → label
```

Где:

- `candidate_news` — новая публикация;
- `cluster_context` — уже известные публикации того же сюжета;
- `label` — экспертная оценка, добавляет ли публикация существенную новую информацию.

Алгоритм формирования выборки:

1. Отсортировать новости по `published_at`.
2. Построить embedding для каждой новости.
3. Обрабатывать новости в хронологическом порядке.
4. Для каждой новости искать ближайший активный сюжетный кластер в temporal window.
5. Если сходство с ближайшим кластером выше порога `T_story`, отнести новость к существующему сюжету.
6. Если подходящего кластера нет — создать новый сюжетный кластер.
7. Внутри кластера вычислить similarity-признаки:
   - `cluster_similarity` — сходство с центроидом кластера;
   - `max_recent_similarity` — максимальное сходство с последними K публикациями кластера;
   - `max_significant_similarity` — максимальное сходство с ранее отмеченными существенными публикациями.
8. На основе ручной разметки сформировать целевую переменную.

### 9.1 Заготовка структуры итогового датасета

На данном этапе, если эмбеддинги и кластеризация ещё не рассчитаны, можно подготовить базовую таблицу с очищенными новостями. Similarity-признаки и `cluster_id` будут добавлены на следующем этапе после запуска semantic encoder и алгоритма кластеризации.

In [ ]:
prepared = df_clean.copy()

# Если идентификатора новости нет, создаём технический.
if ID_COL not in prepared.columns:
    prepared[ID_COL] = np.arange(len(prepared))

# Заготовки будущих колонок. Они будут заполнены после построения эмбеддингов и кластеризации.
for col in ["cluster_id", "cluster_similarity", "max_recent_similarity", "max_significant_similarity", "novelty_score", "label"]:
    if col not in prepared.columns:
        prepared[col] = np.nan

base_output_cols = [ID_COL, DATE_COL]

for col in [SOURCE_COL, TITLE_COL, TEXT_COL, URL_COL]:
    if col in prepared.columns:
        base_output_cols.append(col)

base_output_cols += [
    "text_length",
    "text_num_words",
    "cluster_id",
    "cluster_similarity",
    "max_recent_similarity",
    "max_significant_similarity",
    "novelty_score",
    "label",
]

prepared_dataset = prepared[base_output_cols].copy()
prepared_dataset.head()

### 9.2 Формула novelty score для будущего этапа

После построения кластеров и разметки существенных публикаций можно считать `novelty_score` как непрерывную оценку в диапазоне `[0, 1]`.

Базовая формула для MVP:

```text
novelty_score = (1 - max_significant_similarity) * cluster_similarity
```

Интерпретация:

- высокий `cluster_similarity` означает, что публикация относится к текущему сюжету;
- низкий `max_significant_similarity` означает, что публикация не похожа на уже известные существенные новости;
- произведение этих величин даёт высокую оценку для публикаций, которые остаются внутри сюжета, но добавляют новую информацию.

Для дублей дополнительно используется `max_recent_similarity`: если публикация почти совпадает с одной из последних новостей, она помечается как дубль независимо от novelty score.

In [ ]:
def calculate_novelty_score(cluster_similarity, max_significant_similarity):
    # Базовая формула novelty_score для будущего этапа.
    # Значения similarity ожидаются в диапазоне [0, 1].
    if pd.isna(cluster_similarity) or pd.isna(max_significant_similarity):
        return np.nan
    return (1 - max_significant_similarity) * cluster_similarity

# Пример расчёта:
example_cluster_similarity = 0.88
example_max_significant_similarity = 0.55
calculate_novelty_score(example_cluster_similarity, example_max_significant_similarity)

## 10. Оценка качества разметки

Разметка новизны является частично субъективной: граница между несущественным и существенным обновлением может зависеть от редакционной политики.

Для первой версии предлагается бинарная разметка:

- `not_novel` / `0` — дубль, рерайт или косметическое обновление;
- `novel` / `1` — публикация добавляет новый существенный факт к сюжету.

Для контроля качества разметки предлагается:

1. Подготовить инструкцию для разметчиков с примерами.
2. Часть выборки разметить двумя независимыми разметчиками.
3. Оценить согласованность разметки с помощью Cohen’s Kappa.
4. Разобрать спорные случаи и уточнить инструкцию.

Целевой ориентир: `Cohen’s Kappa >= 0.6`. Если значение ниже, критерии существенности нужно уточнить и повторно разметить спорные примеры.

In [ ]:
# Пример функции для оценки согласованности разметки.
# Работает, когда есть две колонки с разметкой разных аннотаторов.

try:
    from sklearn.metrics import cohen_kappa_score

    def evaluate_annotation_agreement(data, annotator_1_col, annotator_2_col):
        subset = data[[annotator_1_col, annotator_2_col]].dropna()
        if subset.empty:
            raise ValueError("Нет строк, где присутствует разметка обоих аннотаторов.")
        return cohen_kappa_score(subset[annotator_1_col], subset[annotator_2_col])

    print("Функция evaluate_annotation_agreement готова к использованию.")
except ImportError:
    print("sklearn не установлен. Для расчёта Cohen's Kappa установите scikit-learn.")

## 11. Стратегия валидации

Для новостной задачи не подходит случайное разбиение train/test, так как оно может привести к утечке данных:

- публикации одного и того же сюжета могут попасть одновременно в train и test;
- модель или пороги могут быть неявно настроены на события, которые уже присутствуют в обучении.

Основная стратегия валидации — **temporal split**:

- train — более ранний период;
- validation/test — более поздний период.

Такой подход соответствует реальному production-сценарию, где система настраивается на исторических данных и затем применяется к новым публикациям.

Дополнительно можно использовать **group split по `cluster_id`**, чтобы один сюжет не попадал одновременно в обучающую и тестовую выборку.

In [ ]:
def temporal_train_valid_test_split(data, date_col, train_share=0.7, valid_share=0.15):
    # Делит данные по времени без перемешивания.
    data_sorted = data.sort_values(date_col).reset_index(drop=True)
    n = len(data_sorted)

    train_end = int(n * train_share)
    valid_end = int(n * (train_share + valid_share))

    train = data_sorted.iloc[:train_end].copy()
    valid = data_sorted.iloc[train_end:valid_end].copy()
    test = data_sorted.iloc[valid_end:].copy()

    return train, valid, test

train_df, valid_df, test_df = temporal_train_valid_test_split(prepared_dataset, DATE_COL)

split_report = pd.DataFrame({
    "split": ["train", "valid", "test"],
    "rows": [len(train_df), len(valid_df), len(test_df)],
    "date_min": [train_df[DATE_COL].min(), valid_df[DATE_COL].min(), test_df[DATE_COL].min()],
    "date_max": [train_df[DATE_COL].max(), valid_df[DATE_COL].max(), test_df[DATE_COL].max()],
})

split_report

### Вывод по стратегии валидации

Temporal split является предпочтительным вариантом для этой задачи, потому что он моделирует реальный сценарий работы системы с новостным потоком. Случайное разбиение можно использовать только для технических экспериментов, но не для финальной оценки качества.

## 12. Метрики для дальнейшей оценки качества

Так как проект состоит из нескольких этапов, оценивать его нужно на нескольких уровнях.

### 12.1 Качество embedding-модели

На размеченных парах новостей:

- Recall@K;
- MRR@K;
- Precision@K;
- MAP.

Baseline для сравнения:

- TF-IDF + cosine similarity;
- базовая sentence-transformer модель;
- дообученная sentence-transformer модель.

### 12.2 Качество кластеризации

Если есть ручная разметка `story_id`:

- cluster precision;
- ARI;
- NMI;
- V-measure.

Для MVP допустима ручная проверка части кластеров:

```text
cluster_precision = correct_assignments / all_checked_assignments
```

### 12.3 Качество novelty detection

Для бинарной разметки `novel / not_novel`:

- Precision;
- Recall;
- F1;
- Precision@K для top-novel публикаций внутри кластера.

Основной акцент лучше делать на Precision@K и F1 по классу `novel`, так как в продуктовой задаче особенно важно показывать редактору действительно полезные существенные обновления.

## 13. Сохранение подготовленного датасета

На выходе сохраняется очищенный датасет с базовыми техническими признаками и заготовками колонок для следующих этапов: кластеризации, расчёта similarity-признаков и ручной разметки.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
prepared_dataset.to_csv(OUTPUT_PATH, index=False)
print(f"Подготовленный датасет сохранён: {OUTPUT_PATH}")
print(f"Размер сохранённого датасета: {prepared_dataset.shape[0]:,} строк, {prepared_dataset.shape[1]:,} колонок")

## 14. Итоговые выводы

В результате первичного анализа и подготовки данных:

1. Проверена структура исходного новостного датасета.
2. Выполнена базовая проверка качества данных: пропуски, дубли, пустые и короткие тексты, некорректные даты.
3. Выполнена минимальная очистка данных, необходимая для построения эмбеддингов и дальнейшей кластеризации.
4. Проведён EDA, важный для моделирования: длина текстов, распределение по источникам, временное распределение публикаций, корреляции технических признаков.
5. Зафиксировано, что классическое масштабирование сырых текстовых данных не требуется; для cosine similarity будет использоваться L2-нормализация эмбеддингов.
6. Описан алгоритм формирования выборки для задачи novelty detection.
7. Описана стратегия контроля качества разметки.
8. Обоснована temporal validation strategy как наиболее соответствующая реальному сценарию обработки новостного потока.

Итоговый датасет может использоваться на следующих этапах проекта для построения эмбеддингов, инкрементальной кластеризации, расчёта similarity-признаков и настройки novelty scoring.